In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
# training file: https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [2]:
# read it in to inspect it
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("length of dataset in characters: ", len(text))
print(text[0])

length of dataset in characters:  1115394
F


In [ ]:
# create a simple encode/ decode method
stoi = {ch: i for i, ch in enumerate(text)}
itos = {i: ch for i, ch in enumerate(text)}
encode = lambda s: [
    stoi[c] for c in s
]  # encoder: take a string, output a list of integers
decode = lambda l: "".join(
    [itos[i] for i in l]
)  # decoder: take a list of integers, output a string
print(encode("hii there"))
print(decode(encode("hii there")))

# tokenizer eg: openai's tiktoken

[1115378, 1115389, 1115389, 1115385, 1115384, 1115378, 1115374, 1115383, 1115374]
hii there


In [4]:
import torch

data = torch.tensor(
    encode(text), dtype=torch.long
)  # create a one dimension tensor (array)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [5]:
# Let's now split up the data into train and validation sets
n = int(0.9 * len(data))  # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [ ]:
block_size = 8
train_data[: block_size + 1]  # a chunk of training data set
# each chunk has its context

tensor([1111924, 1115389, 1115383, 1115375, 1115384, 1115385, 1110567, 1115389,
        1115384])

In [7]:
# spell out the chunk
x = train_data[:block_size]
y = train_data[1 : block_size + 1]
for t in range(block_size):
    context = x[: t + 1]
    target = y[t]
    print(f"when the context is {context} the target is expected to be {target}")

when the context is tensor([1111924]) the target is expected to be 1115389
when the context is tensor([1111924, 1115389]) the target is expected to be 1115383
when the context is tensor([1111924, 1115389, 1115383]) the target is expected to be 1115375
when the context is tensor([1111924, 1115389, 1115383, 1115375]) the target is expected to be 1115384
when the context is tensor([1111924, 1115389, 1115383, 1115375, 1115384]) the target is expected to be 1115385
when the context is tensor([1111924, 1115389, 1115383, 1115375, 1115384, 1115385]) the target is expected to be 1110567
when the context is tensor([1111924, 1115389, 1115383, 1115375, 1115384, 1115385, 1110567]) the target is expected to be 1115389
when the context is tensor([1111924, 1115389, 1115383, 1115375, 1115384, 1115385, 1110567, 1115389]) the target is expected to be 1115384


In [ ]:
torch.manual_seed(1337)  # fix the seed to reporoduce the data

batch_size = 4  # how many independent sequences will we process in parallel?
block_size = 8  # what is the maximum context length for predictions?


def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == "train" else val_data
    ix = torch.randint(
        len(data) - block_size, (batch_size,)
    )  # 0 to data length - block-size
    print("ix", ix)  # the seed will make ix's result fixed.
    x = torch.stack([data[i : i + block_size] for i in ix])  # add the arbitray offset
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    return x, y


xb, yb = get_batch("train")
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)

print("----")

# b is row number starts from 0
for b in range(batch_size):  # batch dimension
    # t is column number starts from 0, and the training target can be constructed like
    # in each row first x elements from train data will try to perdict the result at the 
    # same row and the same x column
    for t in range(block_size):  # time dimension
        context = xb[b, : t + 1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

ix tensor([ 76049, 234249, 934904, 560986])
inputs:
torch.Size([4, 8])
tensor([[1114446, 1115374, 1115384, 1115366, 1115375, 1115385, 1115378, 1115374],
        [1115334, 1115379, 1115383, 1115385, 1115384, 1115378, 1115387, 1115384],
        [1115390, 1115384, 1115385, 1115384, 1115378, 1115387, 1115384, 1115385],
        [1114933, 1115065, 1115298, 1115299, 1115393, 1115297, 1115385, 1115346]])
targets:
torch.Size([4, 8])
tensor([[1115374, 1115384, 1115366, 1115375, 1115385, 1115378, 1115374, 1115387],
        [1115379, 1115383, 1115385, 1115384, 1115378, 1115387, 1115384, 1115385],
        [1115384, 1115385, 1115384, 1115378, 1115387, 1115384, 1115385, 1115378],
        [1115065, 1115298, 1115299, 1115393, 1115297, 1115385, 1115346, 1115387]])
----
when input is [1114446] the target: 1115374
when input is [1114446, 1115374] the target: 1115384
when input is [1114446, 1115374, 1115384] the target: 1115366
when input is [1114446, 1115374, 1115384, 1115366] the target: 1115375
when inp